## Chunked embeddings → Qdrant (resumes & job postings)

This notebook walks `data/cv-dataset-processed` and `data/job-postings-processed`, turns each JSON into text, splits it into conservative overlapping chunks (target **384 tokens**, well below the model's 512-token context), obtains **1024-dim** embeddings from the local embedding HTTP endpoint (`http://127.0.0.1:8081/v1/embeddings`, model `mxbai-embed-large-v1-f16`), pools chunk vectors using rules analogous to [`STRATEGY.md`](../STRATEGY.md), and writes to a **single shared Qdrant collection**.

- Collection: one shared collection (configured in code).
- Points: each document contributes six points (one per pooling method): `max_score`, `mean_all`, `topk_mean_k5`, `weighted_topk_mean_k5`, `softmax_pooling_alpha10`, `hybrid_max_topk_k5_w0_4`.
- Differentiation: each point payload includes `kind` (`resume` or `job-posting`) plus `domain` and `document_id`, so filtering by source type is straightforward at query time.

**Requirements**

- `./run-llama-servers.sh` (or equivalent): embedding HTTP API on **`127.0.0.1:8081`** (`mxbai-embed-large`).
- Qdrant at `http://localhost:6333` (see [`ollama_connection.ipynb`](ollama_connection.ipynb)).
- **`uv sync`** after pulling deps (includes `tiktoken` for counting tokens; fallback uses a coarse estimate if unavailable).

In [7]:
from __future__ import annotations

import json
import sys
import urllib.request
import uuid
from pathlib import Path
from urllib.error import HTTPError, URLError

import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.http.exceptions import UnexpectedResponse
from qdrant_client.models import Distance, PointStruct, VectorParams

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

# Tiktoken sliding windows (conservative chunk target); falls back to char-based slicing if unavailable.
_HAS_TIKTOKEN = False

try:
    import tiktoken

    _tik_enc = tiktoken.get_encoding("cl100k_base")
    _HAS_TIKTOKEN = True
except ImportError:
    _tik_enc = None


CV_DIR = ROOT / "data" / "cv-dataset-processed"
JOB_DIR = ROOT / "data" / "job-postings-processed"

QDRANT_URL = "http://localhost:6333"
EMBEDDING_API_URLS = (
    "http://127.0.0.1:8081/v1/embeddings",
    "http://localhost:8081/v1/embeddings",
    "http://172.18.80.1:8081/v1/embeddings",
)
EMBED_MODEL = "mxbai-embed-large-v1-f16"
EMBED_DIM = 1024

# Keep a safety margin because local tokenizer estimates can differ from model tokenizer.
MAX_CHUNK_TOKENS = 384
TOKEN_OVERLAP = 64

TOP_K = 5
SOFTMAX_ALPHA = 10.0
HYBRID_W = 0.4

In [8]:
def chunk_text_by_tokens(
    text: str,
    max_tokens: int = MAX_CHUNK_TOKENS,
    overlap_tokens: int = TOKEN_OVERLAP,
) -> list[str]:
    """Windows of up to ``max_tokens``, advancing by ``max_tokens - overlap_tokens``."""

    text = text.strip()
    if not text:
        return []

    overlap_tokens = max(0, min(overlap_tokens, max_tokens - 1))
    step = max(1, max_tokens - overlap_tokens)

    if _HAS_TIKTOKEN and _tik_enc is not None:
        ids = _tik_enc.encode(text)
        chunks: list[str] = []

        for i in range(0, len(ids), step):
            span = ids[i : i + max_tokens]
            if not span:
                continue

            # Re-encode check to avoid decode/encode drift producing oversized chunks.
            while span:
                decoded = _tik_enc.decode(span).strip()
                if not decoded:
                    break
                if len(_tik_enc.encode(decoded)) <= max_tokens:
                    chunks.append(decoded)
                    break
                span = span[:-1]

        first = _tik_enc.decode(ids[:max_tokens]).strip()
        return chunks if chunks else ([first] if first else [])

    # Approximate fallback when tiktoken is not installed (~4 chars / token).
    approx_cpt = 4
    max_chars = max(1, max_tokens * approx_cpt)
    overlap_chars = overlap_tokens * approx_cpt
    stride = max(1, max_chars - overlap_chars)
    chunks = []
    n = len(text)
    start = 0

    while start < n:
        end = min(n, start + max_chars)
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= n:
            break
        start += stride

    return chunks if chunks else [text]

In [9]:
def _l2_normalize(v: np.ndarray) -> np.ndarray:
    n = float(np.linalg.norm(v))
    if n == 0:
        return v
    return v / n


def _numpy_chunk_vectors(embeddings: list[list[float]]) -> np.ndarray:
    return np.asarray(embeddings, dtype=np.float64)


def chunk_salience(emb: np.ndarray) -> float:
    """Proxy for STRATEGY chunk scores when no query exists (L2 norm — model-specific)."""

    return float(np.linalg.norm(emb))


def pool_mean_all(vectors: np.ndarray) -> np.ndarray:
    return _l2_normalize(vectors.mean(axis=0))


def pool_max_score(vectors: np.ndarray, salience: np.ndarray) -> np.ndarray:
    idx = int(np.argmax(salience))
    return _l2_normalize(vectors[idx])


def pool_topk_mean(vectors: np.ndarray, salience: np.ndarray, k: int = TOP_K) -> np.ndarray:
    k_eff = max(1, min(k, len(vectors)))
    order = np.argsort(-salience)[:k_eff]
    return _l2_normalize(vectors[order].mean(axis=0))


def pool_weighted_topk_mean(vectors: np.ndarray, salience: np.ndarray, k: int = TOP_K) -> np.ndarray:
    k_eff = max(1, min(k, len(vectors)))
    order = np.argsort(-salience)[:k_eff]
    weights = np.arange(k_eff, 0, -1, dtype=np.float64)
    weighted = np.average(vectors[order], axis=0, weights=weights)
    return _l2_normalize(weighted)


def pool_softmax(vectors: np.ndarray, salience: np.ndarray, alpha: float = SOFTMAX_ALPHA) -> np.ndarray:
    s = salience.astype(np.float64)
    stabilized = alpha * (s - s.max())
    w = np.exp(stabilized)
    w /= w.sum()
    return _l2_normalize((w[:, None] * vectors).sum(axis=0))


def pool_hybrid(vectors: np.ndarray, salience: np.ndarray, k: int = TOP_K, w: float = HYBRID_W) -> np.ndarray:
    best_idx = int(np.argmax(salience))
    v_max = vectors[best_idx]
    v_topk = pool_topk_mean(vectors, salience, k=k)
    blended = w * v_max + (1.0 - w) * v_topk
    return _l2_normalize(blended)


def build_pooled_vectors(chunk_embeddings: list[list[float]]) -> dict[str, np.ndarray]:
    mats = _numpy_chunk_vectors(chunk_embeddings)

    sal = np.array([chunk_salience(mats[i]) for i in range(len(mats))], dtype=np.float64)

    return {
        "max_score": pool_max_score(mats, sal),
        "mean_all": pool_mean_all(mats),
        "topk_mean_k5": pool_topk_mean(mats, sal, k=TOP_K),
        "weighted_topk_mean_k5": pool_weighted_topk_mean(mats, sal, k=TOP_K),
        "softmax_pooling_alpha10": pool_softmax(mats, sal, alpha=SOFTMAX_ALPHA),
        "hybrid_max_topk_k5_w0_4": pool_hybrid(mats, sal, k=TOP_K, w=HYBRID_W),
    }

In [10]:
def list_json_docs(base: Path) -> list[tuple[str, Path]]:
    rows: list[tuple[str, Path]] = []
    if not base.is_dir():
        return rows

    for domain_dir in sorted(p for p in base.iterdir() if p.is_dir()):
        domain = domain_dir.name
        for jf in sorted(domain_dir.glob("*.json")):
            rows.append((domain, jf))
    return rows


def _doc_label(kind: str, domain: str, stem: str) -> str:
    return f"{kind}-{domain}-{stem}"


def _point_id(kind: str, domain: str, stem: str, pooling_method: str) -> str:
    # Deterministic UUID (accepted by Qdrant) for idempotent upserts.
    key = f"{kind}|{domain}|{stem}|{pooling_method}"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, key))


def _embed_one_chunk(chunk: str) -> list[float]:
    body = json.dumps({"model": EMBED_MODEL, "input": chunk}).encode("utf-8")
    last_error: Exception | None = None

    for api_url in EMBEDDING_API_URLS:
        req = urllib.request.Request(
            api_url,
            data=body,
            headers={"Content-Type": "application/json"},
            method="POST",
        )

        try:
            with urllib.request.urlopen(req, timeout=120) as resp:
                payload = json.loads(resp.read().decode("utf-8"))
        except HTTPError as e:
            # Endpoint reachable but request invalid for this server/model.
            raise RuntimeError(f"embedding API HTTP {e.code} at {api_url}: {e.reason}") from e
        except URLError as e:
            last_error = e
            continue

        data = payload.get("data")
        if not data or not isinstance(data, list):
            raise RuntimeError(f"embedding API returned malformed payload at {api_url}: {payload}")

        vec = data[0].get("embedding")
        if not isinstance(vec, list):
            raise RuntimeError(f"embedding API response missing embedding vector at {api_url}: {payload}")

        return vec

    raise RuntimeError(
        "embedding API connection failed for all endpoints "
        f"{EMBEDDING_API_URLS}. Start embedding server with ./run-llama-servers.sh and ensure port 8081 is reachable. "
        f"Last error: {last_error}"
    )



def embed_chunks(chunks: list[str]) -> list[list[float]]:
    out: list[list[float]] = []
    for chunk in chunks:
        vec = _embed_one_chunk(chunk)

        if len(vec) != EMBED_DIM:
            raise RuntimeError(f"unexpected embedding dimension {len(vec)} (expected {EMBED_DIM})")
        out.append(vec)
    return out


def ensure_collection(client: QdrantClient, name: str, overwrite: bool) -> None:
    names = {c.name for c in client.get_collections().collections}
    exists = name in names

    if exists and overwrite:
        client.delete_collection(name)
        exists = False

    if not exists:
        client.create_collection(
            collection_name=name,
            vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
        )


METHOD_ORDER = (
    "max_score",
    "mean_all",
    "topk_mean_k5",
    "weighted_topk_mean_k5",
    "softmax_pooling_alpha10",
    "hybrid_max_topk_k5_w0_4",
)


def upsert_document_poolings(
    client: QdrantClient,
    collection: str,
    kind: str,
    domain: str,
    stem: str,
    pooled: dict[str, np.ndarray],
    payload_base: dict,
) -> None:
    """One point per pooling method into a shared collection."""

    points: list[PointStruct] = []

    for key in METHOD_ORDER:
        if key not in pooled:
            continue

        vec = pooled[key].astype(float).tolist()
        pid = _point_id(kind, domain, stem, key)

        payload = {
            **payload_base,
            "pooling_method": key,
            "embedding_dim": EMBED_DIM,
        }

        points.append(PointStruct(id=pid, vector=vec, payload=payload))

    if not points:
        return

    client.upsert(collection_name=collection, wait=True, points=points)


def process_documents(
    client: QdrantClient,
    items: list[tuple[str, str, Path]],
    kind_prefix: str,
    collection: str,
    overwrite: bool = False,
):
    """

    Parameters
    ----------
    items :
        tuples of `(domain, json_stem_without_suffix, json_path)`.
    kind_prefix :
        `"resume"` or `"job-posting"`, stored in payload for filtering.
    collection :
        Shared Qdrant collection name.
    """

    try:
        ensure_collection(client, collection, overwrite=overwrite)
    except UnexpectedResponse as e:
        msg = str(e)
        if "No space left on device" in msg:
            raise RuntimeError(
                "Qdrant ran out of disk space while creating collection data. "
                "Free disk space on the Qdrant host volume, then rerun with overwrite=False "
                "to continue from where you stopped."
            ) from e
        raise

    for domain, stem, path in items:
        label = _doc_label(kind_prefix, domain, stem)

        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        text = json.dumps(data, ensure_ascii=False, sort_keys=True, indent=2)

        chunks = chunk_text_by_tokens(text)

        if not chunks:
            print(f"[skip empty] {label}")
            continue

        print(f"[embed] {label} ({len(chunks)} chunks)")
        embeddings = embed_chunks(chunks)

        pooled = build_pooled_vectors(embeddings)

        try:
            upsert_document_poolings(
                client,
                collection,
                kind_prefix,
                domain,
                stem,
                pooled,
                payload_base={
                    "kind": kind_prefix,
                    "domain": domain,
                    "document_id": stem,
                    "json_filename": path.name,
                    "document_key": _doc_label(kind_prefix, domain, stem),
                    "source_path": str(path.relative_to(ROOT)),
                    "chunk_count": len(chunks),
                },
            )
        except UnexpectedResponse as e:
            msg = str(e)
            if "No space left on device" in msg:
                raise RuntimeError(
                    "Qdrant ran out of disk space while writing point data. "
                    "Free disk space on the Qdrant host volume, then rerun with overwrite=False "
                    "to continue from where you stopped."
                ) from e
            raise

In [11]:
# Shared collection for both resumes and job postings.
SHARED_COLLECTION = "resume-job-posting-poolings"

# Keep False for long runs so reruns resume without recreating successful points.
OVERWRITE_COLLECTIONS = False

qdrant = QdrantClient(url=QDRANT_URL)

cv_items = []

for domain, jp in list_json_docs(CV_DIR):
    stem = jp.stem
    cv_items.append((domain, stem, jp))


job_items = []

for domain, jp in list_json_docs(JOB_DIR):
    stem = jp.stem
    job_items.append((domain, stem, jp))


print(f"resume JSON files: {len(cv_items)}")
print(f"job posting JSON files: {len(job_items)}")

resume JSON files: 239
job posting JSON files: 76


In [12]:
process_documents(
    qdrant,
    cv_items,
    "resume",
    collection=SHARED_COLLECTION,
    overwrite=OVERWRITE_COLLECTIONS,
)

[embed] resume-ACCOUNTANT-10554236 (7 chunks)
[embed] resume-ACCOUNTANT-10674770 (3 chunks)
[embed] resume-ACCOUNTANT-11163645 (4 chunks)
[embed] resume-ACCOUNTANT-11759079 (4 chunks)
[embed] resume-ACCOUNTANT-12065211 (4 chunks)
[embed] resume-ACCOUNTANT-12202337 (2 chunks)
[embed] resume-ACCOUNTANT-12338274 (5 chunks)
[embed] resume-ACCOUNTANT-12442909 (6 chunks)
[embed] resume-ACCOUNTANT-12780508 (5 chunks)
[embed] resume-ACCOUNTANT-12802330 (4 chunks)
[embed] resume-ADVOCATE-10186968 (3 chunks)
[embed] resume-ADVOCATE-10344379 (4 chunks)
[embed] resume-ADVOCATE-10659182 (3 chunks)
[embed] resume-ADVOCATE-10818478 (3 chunks)
[embed] resume-ADVOCATE-11174187 (4 chunks)
[embed] resume-ADVOCATE-11188218 (3 chunks)
[embed] resume-ADVOCATE-11773767 (2 chunks)
[embed] resume-ADVOCATE-11963737 (3 chunks)
[embed] resume-ADVOCATE-12171093 (2 chunks)
[embed] resume-ADVOCATE-12544735 (3 chunks)
[embed] resume-AGRICULTURE-10953078 (4 chunks)
[embed] resume-AGRICULTURE-11197262 (3 chunks)
[embed

In [13]:
process_documents(
    qdrant,
    job_items,
    "job-posting",
    collection=SHARED_COLLECTION,
    overwrite=OVERWRITE_COLLECTIONS,
)

[embed] job-posting-AGRICULTURE-00043 (2 chunks)
[embed] job-posting-ARTS-00055 (1 chunks)
[embed] job-posting-AUTOMOBILE-00033 (1 chunks)
[embed] job-posting-AUTOMOBILE-00057 (1 chunks)
[embed] job-posting-AUTOMOBILE-00090 (1 chunks)
[embed] job-posting-BANKING-00046 (1 chunks)
[embed] job-posting-BUSINESS-DEVELOPMENT-00081 (1 chunks)
[embed] job-posting-CONSTRUCTION-00031 (1 chunks)
[embed] job-posting-CONSULTANT-00013 (1 chunks)
[embed] job-posting-CONSULTANT-00030 (1 chunks)
[embed] job-posting-CONSULTANT-00086 (2 chunks)
[embed] job-posting-DESIGNER-00019 (1 chunks)
[embed] job-posting-DESIGNER-00076 (1 chunks)
[embed] job-posting-ENGINEERING-00056 (2 chunks)
[embed] job-posting-FINANCE-00000 (2 chunks)
[embed] job-posting-FINANCE-00006 (1 chunks)
[embed] job-posting-FINANCE-00017 (2 chunks)
[embed] job-posting-FINANCE-00029 (1 chunks)
[embed] job-posting-FINANCE-00053 (1 chunks)
[embed] job-posting-FINANCE-00069 (1 chunks)
[embed] job-posting-FINANCE-00073 (1 chunks)
[embed] job-